# FileMan Data Dictionary Analysis
### VistA VEHU — Interactive Analysis Notebook

This notebook follows the **FileMan Data Dictionary Analysis Guide** phase by phase.
Run cells in order. Each phase builds on variables defined in earlier phases.

**Before running:** ensure the container is started and this notebook is opened via:
```bash
source /usr/local/etc/ydb_env_set
cd /opt/vista-fm-browser
source .venv/bin/activate
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```
Then open `http://localhost:8888` in a browser on the host.

In [ ]:
# ── Environment check ──────────────────────────────────────────────────────────
import os
import sys

print(f"Python:      {sys.version.split()[0]}")
print(
    f"ydb_gbldir:  {os.environ.get('ydb_gbldir', '⚠ NOT SET — run: source /usr/local/etc/ydb_env_set')}"
)
print(f"ydb_dist:    {os.environ.get('ydb_dist', '⚠ NOT SET')}")
print(f"Working dir: {os.getcwd()}")

## Standard Imports
Run this cell once; all phases use these names.

In [ ]:
%matplotlib inline
import collections
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vista_fm_browser.connection import YdbConnection
from vista_fm_browser.data_dictionary import DataDictionary
from vista_fm_browser.file_reader import FileReader
from vista_fm_browser.inventory import FileInventory

# Output directory — created if missing
OUT = Path("~/data/vista-fm-browser/output/").expanduser()
OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUT}")

---
## Phase 1 — Scope Survey
**Goal:** Know the total size of the problem in five numbers before touching any field.

**Expected scope on VEHU:** ~2,500 files · ~90 packages · ~600 unpackaged files

### 1.0 File #1 — the file registry itself
File #1 is the FILE file (`^DIC`) — the meta-file that registers every other FileMan file.

In [ ]:
# Inspect File #1 schema via DataDictionary
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    fd1 = dd.get_file(1)
    print(f"File 1: {fd1.label}  global=^DIC  fields={fd1.field_count}\n")
    for field_num, fld in sorted(fd1.fields.items()):
        ptr = f" → File #{fld.pointer_file}" if fld.pointer_file else ""
        print(f"  {field_num:7.4f}  {fld.label:30s}  {fld.datatype_name}{ptr}")

In [ ]:
# Read File #1 entries through FileReader — same data FileInventory reads
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    reader = FileReader(conn, dd)
    total = reader.count_entries(1)
    print(f"File #1 has {total} entries (registered FileMan files)\n")
    print("First 20 files in the registry:")
    for entry in reader.iter_entries(1, limit=20):
        name = entry.fields.get(0.01, "").strip()
        gl_name = entry.fields.get(1, "").strip()
        pkg_ien = entry.fields.get(4, "").strip()
        print(f"  IEN {entry.ien:>6s}  {name:40s}  ^{gl_name}  pkg_ien={pkg_ien}")

In [ ]:
# Cross-references on File #1 (B-index etc.)
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    xrefs = dd.list_cross_refs(1)
    print(f"File #1 cross-references ({len(xrefs)} total):")
    for xref in xrefs:
        print(f"  '{xref.name}'  ({xref.xref_type})  — {xref.description[:60]}")

### 1.1 Package and file counts

In [ ]:
with YdbConnection.connect() as conn:
    fi = FileInventory(conn)
    fi.load()

s = fi.summary()
print(f"Files total:      {s['total_files']}")
print(f"Packages total:   {s['total_packages']}")
print(f"Unpackaged files: {s['unpackaged_files']}")

grouped = fi.files_by_package()
by_count = sorted(
    ((k, len(v)) for k, v in grouped.items() if k != "(unpackaged)"),
    key=lambda x: -x[1],
)
print("\nTop 20 packages by file count:")
for name, count in by_count[:20]:
    print(f"  {name:45s} {count:4d} files")

### 1.2 Total field count

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    all_files = dd.list_files()
    total_fields = 0
    for file_num, _label in all_files:
        fd = dd.get_file(file_num)
        if fd:
            total_fields += fd.field_count

print(f"Total files:     {len(all_files)}")
print(f"Total fields:    {total_fields:,}")
print(f"Avg fields/file: {total_fields / len(all_files):.1f}")

### 1.3 Field type distribution — the variety picture

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    type_counts: dict[str, int] = collections.Counter()
    type_names: dict[str, str] = {}
    for file_num, _label in dd.list_files():
        fd = dd.get_file(file_num)
        if not fd:
            continue
        for fld in fd.fields.values():
            type_counts[fld.datatype_code] += 1
            type_names[fld.datatype_code] = fld.datatype_name

total = sum(type_counts.values())
print(f"{'Type':6s}  {'Name':22s}  {'Count':>7s}  {'%':>6s}")
print("-" * 50)
for code_, count in type_counts.most_common():
    name = type_names.get(code_, code_)
    pct = 100 * count / total
    print(f"  {code_:4s}  {name:22s}  {count:7,d}  {pct:5.1f}%")
print(f"  {'TOTAL':4s}  {'':22s}  {total:7,d}  100.0%")

### 1.4 Export inventory

In [ ]:
with YdbConnection.connect() as conn:
    fi = FileInventory(conn)
    fi.load()
    out = fi.export_json(OUT)
    print(f"Inventory written to {out}")

### Phase 1 — Visualization

In [ ]:
# Top packages bar chart + field type pie
# Requires: fi (FileInventory), type_counts, type_names from cells above

s = fi.summary()
top = s["top_packages_by_file_count"][:20]
names_ = [r["name"][:30] for r in reversed(top)]
counts_ = [r["file_count"] for r in reversed(top)]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
bars = ax.barh(names_, counts_, color="steelblue")
ax.bar_label(bars, padding=3, fontsize=8)
ax.set_xlabel("File count")
ax.set_title(
    f"Top 20 VistA Packages by File Count\n"
    f"(total: {s['total_files']} files, {s['total_packages']} packages)"
)
ax.tick_params(axis="y", labelsize=8)

ax2 = axes[1]
labels_ = [
    f"{code_}\n{type_names.get(code_, '')} ({cnt:,})"
    for code_, cnt in type_counts.most_common()
]
sizes_ = [cnt for _, cnt in type_counts.most_common()]
explode = [0.05] * len(sizes_)
ax2.pie(
    sizes_,
    labels=labels_,
    explode=explode,
    autopct="%1.1f%%",
    startangle=140,
    textprops={"fontsize": 8},
)
ax2.set_title("Field Type Distribution across All Files")

plt.tight_layout()
fig.savefig(OUT / "phase1_scope.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT / 'phase1_scope.png'}")

---
## Phase 2 — Volume Survey
**Goal:** Find out where the actual data lives. Separate heavyweight clinical files from small configuration files.

> ⚠ **This cell is slow** — it counts entries in every file. On VEHU expect 5–20 minutes.

### 2.1 Entry counts for all files with data

In [ ]:
with YdbConnection.connect() as conn:
    reader = FileReader(conn, DataDictionary(conn))
    fi = FileInventory(conn)
    fi.load()

    volume: list[tuple[int, float, str]] = []
    for fr in fi.list_files():
        count = reader.count_entries(fr.file_number)
        if count > 0:
            volume.append((count, fr.file_number, fr.label))

volume.sort(reverse=True)
print(f"Files with data: {len(volume)} of {len(fi.list_files())}")
print("\nTop 40 files by entry count:")
file_pkg = {fr.file_number: fr.package_name or "?" for fr in fi.list_files()}
for count, num, label in volume[:40]:
    print(f"  {num:8.2f}  {label:40s}  {count:>10,}  [{file_pkg.get(num, '?')}]")

# Save to disk
out_vol = OUT / "file_volume.json"
out_vol.write_text(
    json.dumps(
        [{"file_number": n, "label": l, "entry_count": c} for c, n, l in volume],
        indent=2,
    )
)
print(f"\nVolume data written to {out_vol}")

### 2.2 Volume tiers

In [ ]:
tiers = {
    "massive (>100K entries)": [],
    "large (10K–100K)": [],
    "medium (1K–10K)": [],
    "small (100–1K)": [],
    "tiny (<100)": [],
    "empty (0)": [],
}
for count, num, label in volume:
    if count == 0:
        tiers["empty (0)"].append((num, label))
    elif count < 100:
        tiers["tiny (<100)"].append((num, label, count))
    elif count < 1_000:
        tiers["small (100–1K)"].append((num, label, count))
    elif count < 10_000:
        tiers["medium (1K–10K)"].append((num, label, count))
    elif count < 100_000:
        tiers["large (10K–100K)"].append((num, label, count))
    else:
        tiers["massive (>100K entries)"].append((num, label, count))

for tier, files in tiers.items():
    print(f"{tier}: {len(files)} files")

### 2.3 Data density — bytes per entry estimate

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    reader = FileReader(conn, dd)
    for count, num, label in volume[:20]:
        entry = reader.get_entry(num, "1")
        if not entry:
            continue
        raw_size = sum(len(v) for v in entry.raw_nodes.values())
        field_count = dd.get_file(num).field_count if dd.get_file(num) else 0
        print(
            f"  {num:8.2f}  {label:35s}  "
            f"{count:>8,} entries  "
            f"{raw_size:4d} bytes/entry (sample)  "
            f"{field_count:3d} fields"
        )

### Phase 2 — Visualization

In [ ]:
TIER_COLORS = {
    "massive": "#d62728",
    "large": "#ff7f0e",
    "medium": "#2ca02c",
    "small": "#1f77b4",
    "tiny": "#aec7e8",
}


def tier_color(count):
    if count >= 100_000:
        return TIER_COLORS["massive"]
    if count >= 10_000:
        return TIER_COLORS["large"]
    if count >= 1_000:
        return TIER_COLORS["medium"]
    if count >= 100:
        return TIER_COLORS["small"]
    return TIER_COLORS["tiny"]


top50 = [(c, n, l) for c, n, l in volume if c > 0][:50]
labels_p = [f"{l[:35]} (#{n:.0f})" for _, n, l in reversed(top50)]
counts_p = [c for c, _, _ in reversed(top50)]
colors_p = [tier_color(c) for c in counts_p]

fig, ax = plt.subplots(figsize=(12, max(10, len(top50) * 0.28)))
ax.barh(labels_p, counts_p, color=colors_p, log=True)
ax.set_xlabel("Entry count (log scale)")
ax.set_title(f"Top {len(top50)} FileMan Files by Entry Count")
ax.tick_params(axis="y", labelsize=7)
patches = [mpatches.Patch(color=v, label=k) for k, v in TIER_COLORS.items()]
ax.legend(handles=patches, loc="lower right", fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "phase2_volume.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT / 'phase2_volume.png'}")

In [ ]:
# Volume tier summary as pandas DataFrame + CSV export
rows_v = []
for count, num, label in volume:
    if count == 0:
        tier = "empty"
    elif count < 100:
        tier = "tiny"
    elif count < 1_000:
        tier = "small"
    elif count < 10_000:
        tier = "medium"
    elif count < 100_000:
        tier = "large"
    else:
        tier = "massive"
    rows_v.append(
        {"file_number": num, "label": label, "entry_count": count, "tier": tier}
    )

df_vol = pd.DataFrame(rows_v)
print(df_vol.groupby("tier")["file_number"].count().rename("files"))
df_vol.to_csv(OUT / "file_volume.csv", index=False)
print(f"\nVolume CSV: {OUT / 'file_volume.csv'}")

---
## Phase 3 — Structural Topology
**Goal:** Map the pointer graph — how files reference each other.

> ⚠ **3.1 is slow** — reads every `^DD` entry. Run once; result cached to `all_fields.json`.

### 3.1 Build the full schema (one pass)

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    fi = FileInventory(conn)
    fi.load()

    pkg_by_file: dict[float, str] = {
        fr.file_number: (fr.package_name or "(unpackaged)") for fr in fi.list_files()
    }

    schema: list[dict] = []
    for file_num, file_label in dd.list_files():
        fd = dd.get_file(file_num)
        if not fd:
            continue
        for field_num, fld in fd.fields.items():
            schema.append(
                {
                    "file_number": file_num,
                    "file_label": file_label,
                    "package": pkg_by_file.get(file_num, "(unpackaged)"),
                    "field_number": field_num,
                    "field_label": fld.label,
                    "datatype_code": fld.datatype_code,
                    "datatype_name": fld.datatype_name,
                    "pointer_file": fld.pointer_file,
                    "set_values": fld.set_values,
                }
            )

out_schema = OUT / "all_fields.json"
out_schema.write_text(json.dumps(schema, indent=2, default=str))
print(f"Schema: {len(schema):,} fields across {len(dd.list_files()):,} files")
print(f"Written to {out_schema}")

### 3.2 Pointer graph — hub file identification

In [ ]:
pointer_fields = [r for r in schema if r["datatype_code"] == "P" and r["pointer_file"]]

inbound: dict[float, set[float]] = {}
for r in pointer_fields:
    tgt = r["pointer_file"]
    inbound.setdefault(tgt, set()).add(r["file_number"])

print(f"Pointer fields:  {len(pointer_fields):,}")
print(f"Unique targets:  {len(inbound):,} files referenced by pointers")
print("\nTop 30 hub files (most-referenced):")
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    for tgt, srcs in sorted(inbound.items(), key=lambda x: -len(x[1]))[:30]:
        fd = dd.get_file(tgt)
        label = fd.label if fd else "?"
        print(f"  File {tgt:8.2f}  {label:40s}  ← {len(srcs):3d} files")

### 3.3 Pointer graph — outbound density per file

In [ ]:
outbound: dict[float, set[float]] = {}
for r in pointer_fields:
    outbound.setdefault(r["file_number"], set()).add(r["pointer_file"])

print("Top 20 files by outbound pointer count (most FK-rich):")
for file_num, targets in sorted(outbound.items(), key=lambda x: -len(x[1]))[:20]:
    fd_label = next(
        (r["file_label"] for r in schema if r["file_number"] == file_num), "?"
    )
    pkg = pkg_by_file.get(file_num, "?")
    print(
        f"  File {file_num:8.2f}  {fd_label:40s}  → {len(targets):3d} targets  [{pkg}]"
    )

### 3.4 Export the pointer graph

In [ ]:
edges = [
    {
        "from_file": r["file_number"],
        "from_label": r["file_label"],
        "from_pkg": r["package"],
        "field_num": r["field_number"],
        "field_label": r["field_label"],
        "to_file": r["pointer_file"],
    }
    for r in pointer_fields
]
out_edges = OUT / "pointer_graph.json"
out_edges.write_text(json.dumps(edges, indent=2, default=str))
print(f"Pointer graph: {len(edges):,} edges → {out_edges}")

### 3.5 Variable pointer fields (polymorphic FKs)

In [ ]:
variable_pointers = [r for r in schema if r["datatype_code"] == "V"]
print(f"Variable pointer fields: {len(variable_pointers)}")
vp_by_file = collections.Counter(r["file_number"] for r in variable_pointers)
for file_num, count in vp_by_file.most_common(10):
    label = next((r["file_label"] for r in schema if r["file_number"] == file_num), "?")
    print(f"  File {file_num:.2f}  {label:40s}  {count} variable pointer fields")

### 3.6 Multiple (sub-file) depth map

In [ ]:
multiple_fields = [r for r in schema if r["datatype_code"] == "M"]
print(f"MULTIPLE fields (sub-files): {len(multiple_fields)}")
multiples_per_file = collections.Counter(r["file_number"] for r in multiple_fields)
print("Files with most MULTIPLE fields (most complex hierarchy):")
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    for file_num, count in multiples_per_file.most_common(15):
        fd = dd.get_file(file_num)
        label = fd.label if fd else "?"
        print(f"  File {file_num:8.2f}  {label:40s}  {count} sub-files")

### Phase 3 — Visualization

In [ ]:
import networkx as nx

# Hub-file subgraph
G = nx.DiGraph()
for r in pointer_fields:
    G.add_edge(r["file_number"], r["pointer_file"], field_label=r["field_label"])

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

hub_file_nums = {tgt for tgt, srcs in inbound.items() if len(srcs) >= 10}
neighbors = set()
for h in hub_file_nums:
    neighbors.update(G.predecessors(h))
H = G.subgraph(hub_file_nums | neighbors).copy()

with YdbConnection.connect() as conn:
    fi2 = FileInventory(conn)
    fi2.load()
    file_labels_map = {
        fr.file_number: f"#{fr.file_number:.0f}\n{fr.label[:18]}"
        for fr in fi2.list_files()
    }

sizes = [
    3000 + 500 * len(inbound.get(n, set())) if n in hub_file_nums else 600
    for n in H.nodes()
]
colors_g = ["#d62728" if n in hub_file_nums else "#aec7e8" for n in H.nodes()]

pos = nx.spring_layout(H, k=2.5, seed=42)
fig, ax = plt.subplots(figsize=(20, 14))
nx.draw_networkx_nodes(H, pos, node_size=sizes, node_color=colors_g, alpha=0.85, ax=ax)
nx.draw_networkx_labels(
    H,
    pos,
    labels={n: file_labels_map.get(n, str(n)) for n in H.nodes()},
    font_size=6,
    ax=ax,
)
nx.draw_networkx_edges(
    H, pos, alpha=0.2, arrows=True, arrowsize=8, edge_color="gray", ax=ax
)
ax.set_title(
    "FileMan Pointer Graph — Hub Files (≥10 inbound) + Neighbors\nRed = hub, Blue = source"
)
ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "phase3_pointer_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT / 'phase3_pointer_graph.png'}")

In [ ]:
# Package-to-package cross-pointer matrix
pkg_pairs: dict[tuple[str, str], int] = collections.Counter()
for r in pointer_fields:
    sp = pkg_by_file.get(r["file_number"], "(unpackaged)")
    tp = pkg_by_file.get(r["pointer_file"], "(unpackaged)")
    if sp != tp:
        pkg_pairs[(sp, tp)] += 1

top_pkgs = [
    p
    for p, _ in collections.Counter(
        {
            p: sum(v for (s, t), v in pkg_pairs.items() if s == p or t == p)
            for p in set(s for s, _ in pkg_pairs) | set(t for _, t in pkg_pairs)
        }
    ).most_common(15)
]

matrix = pd.DataFrame(0, index=top_pkgs, columns=top_pkgs)
for (src, tgt), cnt in pkg_pairs.items():
    if src in top_pkgs and tgt in top_pkgs:
        matrix.loc[src, tgt] += cnt

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(matrix.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(top_pkgs)))
ax.set_yticks(range(len(top_pkgs)))
ax.set_xticklabels([p[:20] for p in top_pkgs], rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([p[:20] for p in top_pkgs], fontsize=7)
plt.colorbar(im, ax=ax, label="Cross-package pointer count")
ax.set_title("Cross-Package Pointer Dependency Matrix")
plt.tight_layout()
fig.savefig(OUT / "phase3_pkg_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Export as Graphviz DOT (render on host: dot -Tsvg phase3_pointer_graph.dot -o graph.svg)
dot_lines = [
    "digraph vista_pointers {",
    "  rankdir=LR;",
    "  node [shape=box fontsize=9];",
]
for n in hub_file_nums:
    lbl = file_labels_map.get(n, str(n)).replace("\n", " ")
    dot_lines.append(f'  "{n}" [label="{lbl}" style=filled fillcolor=salmon];')
for u, v in H.edges():
    dot_lines.append(f'  "{u}" -> "{v}";')
dot_lines.append("}")
(OUT / "phase3_pointer_graph.dot").write_text("\n".join(dot_lines))
print("DOT file written → render with: dot -Tsvg phase3_pointer_graph.dot -o graph.svg")

---
## Phase 4 — Data Variety and Naming Analysis
**Goal:** Understand SET-OF-CODES value sets, naming consistency, and shared vocabulary.

### 4.1 SET-OF-CODES inventory — all enumerated value sets

In [ ]:
set_fields = [r for r in schema if r["datatype_code"] == "S" and r["set_values"]]
print(f"SET-OF-CODES fields with defined values: {len(set_fields)}")


def canon_set(sv: dict) -> frozenset:
    return frozenset((k.strip().upper(), v.strip().upper()) for k, v in sv.items())


seen_sets: dict[frozenset, list[dict]] = {}
for r in set_fields:
    key = canon_set(r["set_values"])
    seen_sets.setdefault(key, []).append(r)

shared = [(key, grp) for key, grp in seen_sets.items() if len(grp) >= 5]
shared.sort(key=lambda x: -len(x[1]))
print(f"\nValue sets shared across ≥5 fields ({len(shared)} total):")
for key, group in shared[:15]:
    sample_codes = dict(list(key)[:4])
    print(f"\n  codes={sample_codes}")
    print(
        f"  used in {len(group)} fields across {len(set(r['file_number'] for r in group))} files:"
    )
    for r in group[:3]:
        print(
            f"    File {r['file_number']:.2f} {r['file_label']:30s} · {r['field_label']}"
        )

### 4.2 Boolean equivalents — YES/NO patterns

In [ ]:
boolean_patterns = [
    frozenset({("Y", "YES"), ("N", "NO")}),
    frozenset({("1", "YES"), ("0", "NO")}),
    frozenset({("A", "ACTIVE"), ("I", "INACTIVE")}),
    frozenset({("1", "ACTIVE"), ("0", "INACTIVE")}),
]
for pattern in boolean_patterns:
    matches = seen_sets.get(pattern, [])
    if matches:
        codes = dict(list(pattern))
        print(
            f"  {codes}: {len(matches)} fields in {len(set(r['file_number'] for r in matches))} files"
        )

### 4.3 Label frequency — shared vocabulary

In [ ]:
label_counter = collections.Counter(r["field_label"].strip().upper() for r in schema)
print("Top 40 field labels across all files:")
for label, count in label_counter.most_common(40):
    print(f"  {label:45s}  {count:5,}")

### 4.4 Label-type consistency — same label, different types

In [ ]:
label_groups: dict[str, list[dict]] = {}
for r in schema:
    label_groups.setdefault(r["field_label"].strip().upper(), []).append(r)

inconsistent = []
for label, rows in label_groups.items():
    if len(rows) < 5:
        continue
    types = set(r["datatype_code"] for r in rows)
    if len(types) > 1:
        type_dist = collections.Counter(r["datatype_code"] for r in rows)
        inconsistent.append(
            {
                "label": label,
                "occurrences": len(rows),
                "types": dict(type_dist),
                "files": len(set(r["file_number"] for r in rows)),
            }
        )

inconsistent.sort(key=lambda x: -x["occurrences"])
print(
    f"Labels with same name but inconsistent types (≥5 occurrences): {len(inconsistent)}"
)
for item in inconsistent[:20]:
    print(
        f"  {item['label']:40s}  in {item['occurrences']:4d} fields  types={item['types']}"
    )

### 4.5 Canonical field positions

In [ ]:
canonical_positions = {
    0.01: "PRIMARY NAME/IDENTIFIER",
    0.02: "CATEGORY/TYPE",
    0.03: "PARENT REFERENCE or DATE",
    0.05: "SEX or SECONDARY IDENTIFIER",
    0.07: "STATUS",
    0.09: "SSN or UNIQUE ID",
    1.0: "SECONDARY CONTENT",
    99.0: "CLASS/CATEGORY",
}
print("Canonical field position analysis:")
for pos, concept in canonical_positions.items():
    hits = [r for r in schema if abs(r["field_number"] - pos) < 0.001]
    if not hits:
        continue
    type_dist = collections.Counter(r["datatype_code"] for r in hits)
    top_labels = collections.Counter(r["field_label"].upper() for r in hits)
    print(f"\n  Field {pos:.2f}  ({concept}):  {len(hits)} definitions")
    print(f"    Types:  {dict(type_dist.most_common(4))}")
    print(f"    Labels: {dict(top_labels.most_common(5))}")

### Phase 4 — Visualization

In [ ]:
# Chart 1: Top 40 field label frequencies
top_labels_list = label_counter.most_common(40)
lnames = [l for l, _ in reversed(top_labels_list)]
lcounts = [c for _, c in reversed(top_labels_list)]

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(lnames, lcounts, color="steelblue")
ax.set_xlabel("Number of fields with this label (across all files)")
ax.set_title("Top 40 Field Labels — Shared Vocabulary Across VistA Packages")
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
fig.savefig(OUT / "phase4_label_frequency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 2: Label-type inconsistency heatmap
top_inconsistent = sorted(inconsistent, key=lambda x: -x["occurrences"])[:30]
all_types = ["F", "P", "S", "D", "N", "M", "W", "C", "K", "V"]
rows_data = []
for item in top_inconsistent:
    total_i = item["occurrences"]
    rows_data.append([item["types"].get(t, 0) / total_i for t in all_types])

mat = np.array(rows_data)
ylabels = [item["label"][:35] for item in top_inconsistent]

fig2, ax2 = plt.subplots(figsize=(12, 10))
im = ax2.imshow(mat, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax2.set_xticks(range(len(all_types)))
ax2.set_yticks(range(len(ylabels)))
ax2.set_xticklabels(all_types, fontsize=9)
ax2.set_yticklabels(ylabels, fontsize=7)
plt.colorbar(im, ax=ax2, label="Fraction of occurrences with this type")
ax2.set_title("Label-Type Inconsistency — Same Label, Different Types")
plt.tight_layout()
fig2.savefig(OUT / "phase4_label_type_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 3: SET value Jaccard similarity matrix
top_sets = [
    (key, grp) for key, grp in sorted(seen_sets.items(), key=lambda x: -len(x[1]))[:30]
]
n = len(top_sets)
sim = np.zeros((n, n))
for i, (ki, _) in enumerate(top_sets):
    for j, (kj, _) in enumerate(top_sets):
        if i == j:
            sim[i, j] = 1.0
        else:
            inter = len(ki & kj)
            union = len(ki | kj)
            sim[i, j] = inter / union if union else 0

set_labels_plot = [
    ", ".join(f"{k}={v}" for k, v in list(dict(key))[:2])[:25] for key, _ in top_sets
]
fig3, ax3 = plt.subplots(figsize=(12, 11))
im3 = ax3.imshow(sim, cmap="Blues", vmin=0, vmax=1)
ax3.set_xticks(range(n))
ax3.set_yticks(range(n))
ax3.set_xticklabels(set_labels_plot, rotation=45, ha="right", fontsize=6)
ax3.set_yticklabels(set_labels_plot, fontsize=6)
plt.colorbar(im3, ax=ax3, label="Jaccard similarity")
ax3.set_title(
    "SET Value Set Similarity (top 30)\n1.0 = identical value sets under different labels"
)
plt.tight_layout()
fig3.savefig(OUT / "phase4_set_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Phase 5 — Schema Deep Dive (per file or package)
**Goal:** Full field-level schema for individual files — storage layout, validation, help text, last-edit dates.

Change `file_num` to inspect a different file.

### 5.1 Full field attributes

In [ ]:
file_num = 2  # PATIENT — change to any file number

with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    fd = dd.get_file(file_num)
    print(f"File {file_num}: {fd.label}  ({fd.field_count} fields)\n")

    extended = []
    for field_num in sorted(fd.fields.keys()):
        fa = dd.get_field_attributes(file_num, field_num)
        if fa is None:
            continue
        extended.append(
            {
                "field": fa.field_number,
                "label": fa.label,
                "type": fa.datatype_name,
                "storage": fa.global_subscript,
                "pointer_file": fa.pointer_file,
                "set_values": fa.set_values,
                "help_prompt": (fa.help_prompt or "")[:60],
                "has_description": bool(fa.description),
                "input_transform": bool(fa.input_transform),
                "last_edited": fa.last_edited,
            }
        )
        print(
            f"  {fa.field_number:8.4f}  {fa.label:30s}  {fa.datatype_name:15s}  "
            f"loc={fa.global_subscript:8s}"
        )

### 5.2 Storage layout — zero-node density

In [ ]:
node_map: dict[str, list[str]] = defaultdict(list)
for row in extended:
    loc = row["storage"]
    if ";" in loc:
        node, piece = loc.split(";", 1)
        node_map[node].append(f"{row['field']:.4f} {row['label']}")

print(f"Storage nodes for {fd.label}:")
for node in sorted(node_map.keys()):
    fields_here = node_map[node]
    print(
        f"  Node {node:6s}: {len(fields_here)} fields — "
        f"{', '.join(fields_here[:4])}" + ("..." if len(fields_here) > 4 else "")
    )

### 5.3 Cross-reference inventory

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    refs = dd.list_cross_refs(file_num)
    print(f"Cross-references for {fd.label}:")
    for ref in refs:
        print(f"  '{ref.name}' ({ref.xref_type})  {ref.description[:60]}")

### 5.4 Per-package schema batch export

In [ ]:
# Exports one JSON per package to output/packages/
with YdbConnection.connect() as conn:
    fi = FileInventory(conn)
    fi.load()
    dd = DataDictionary(conn)
    pkg_out = OUT / "packages"
    pkg_out.mkdir(parents=True, exist_ok=True)
    for pkg_name, files in fi.files_by_package().items():
        pkg_schema = []
        for fr in files:
            fd2 = dd.get_file(fr.file_number)
            if not fd2:
                continue
            for fn2, fld2 in fd2.fields.items():
                pkg_schema.append(
                    {
                        "file_number": fr.file_number,
                        "file_label": fr.label,
                        "field_number": fn2,
                        "field_label": fld2.label,
                        "type_code": fld2.datatype_code,
                        "type_name": fld2.datatype_name,
                        "pointer_file": fld2.pointer_file,
                    }
                )
        safe = pkg_name.replace("/", "_").replace(" ", "_").lower()[:40]
        (pkg_out / f"{safe}.json").write_text(
            json.dumps(pkg_schema, indent=2, default=str)
        )
    print(f"Package schemas written to {pkg_out}")

### Phase 5 — Visualization

In [ ]:
# Documentation completeness heatmap
attr_labels_5 = [
    "Description",
    "Input Transform",
    "Set Values",
    "Last Edited",
    "Help Prompt",
]


def has_val(row, key):
    v = row.get(key)
    return 0 if (v is None or v == "" or v is False) else 1


def has_set(row):
    return 1 if (row.get("set_values") or {}) else 0


mat5 = np.array(
    [
        [
            has_val(r, "has_description"),
            has_val(r, "input_transform"),
            has_set(r),
            1 if r.get("last_edited") else 0,
            has_val(r, "help_prompt"),
        ]
        for r in extended
    ],
    dtype=float,
)

field_names_5 = [f"{r['field']:.4f} {r['label'][:30]}" for r in extended]
fig5, ax5 = plt.subplots(figsize=(10, max(6, len(field_names_5) * 0.22)))
im5 = ax5.imshow(mat5, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax5.set_xticks(range(len(attr_labels_5)))
ax5.set_yticks(range(len(field_names_5)))
ax5.set_xticklabels(attr_labels_5, fontsize=9)
ax5.set_yticklabels(field_names_5, fontsize=6)
plt.colorbar(im5, ax=ax5, label="Present (1) / Absent (0)", shrink=0.4)
ax5.set_title(
    f"Field Documentation Completeness — {fd.label} (File #{file_num})\nGreen = present, Red = missing"
)
plt.tight_layout()
fig5.savefig(OUT / f"phase5_schema_{int(file_num)}.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Phase 6 — Data Coverage Analysis
**Goal:** Which defined fields are actually populated vs. defined-but-unused?

Change `file_num` or `SAMPLE` as needed.

In [ ]:
file_num = 2  # PATIENT
SAMPLE = 500

with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    reader = FileReader(conn, dd)
    fd = dd.get_file(file_num)

    field_hits: dict[float, int] = defaultdict(int)
    count = 0
    for entry in reader.iter_entries(file_num, limit=SAMPLE):
        count += 1
        for fn, val in entry.fields.items():
            if val.strip():
                field_hits[fn] += 1

print(f"Coverage for {fd.label} (n={count}):")
print(f"{'Field':8s}  {'Label':30s}  {'%Pop':>6s}  {'Type':15s}")
print("-" * 65)
for fn in sorted(fd.fields.keys()):
    fld = fd.fields[fn]
    hits = field_hits.get(fn, 0)
    pct = 100 * hits / count if count else 0
    bar = "▓" * int(pct / 10)
    print(f"  {fn:6.4f}  {fld.label:30s}  {pct:5.1f}%  {fld.datatype_name:15s}  {bar}")

### Phase 6 — Visualization

In [ ]:
rows_cov = []
for fn in sorted(fd.fields.keys()):
    fld = fd.fields[fn]
    hits = field_hits.get(fn, 0)
    pct = 100 * hits / count if count else 0
    rows_cov.append((pct, fn, fld.label, fld.datatype_name))

rows_cov.sort(key=lambda x: x[0])


def cov_color(pct):
    if pct >= 80:
        return "#2ca02c"
    if pct >= 20:
        return "#ff7f0e"
    return "#d62728"


labels_cov = [f"{num:.4f} {lbl[:28]} ({dtype[:3]})" for _, num, lbl, dtype in rows_cov]
pcts_cov = [p for p, *_ in rows_cov]
bar_colors_cov = [cov_color(p) for p in pcts_cov]

fig6, ax6 = plt.subplots(figsize=(11, max(8, len(labels_cov) * 0.22)))
ax6.barh(labels_cov, pcts_cov, color=bar_colors_cov)
ax6.set_xlim(0, 105)
ax6.set_xlabel("% of sampled entries where field is populated")
ax6.set_title(
    f"Field Coverage — {fd.label} (File #{file_num})\n"
    f"n={count}  |  Green ≥80%, Orange 20–80%, Red <20%"
)
ax6.tick_params(axis="y", labelsize=7)
ax6.axvline(x=80, color="green", linestyle="--", alpha=0.4, linewidth=0.8)
ax6.axvline(x=20, color="orange", linestyle="--", alpha=0.4, linewidth=0.8)
plt.tight_layout()
fig6.savefig(OUT / f"phase6_coverage_{int(file_num)}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Multi-file coverage comparison
key_files = [(2, "PATIENT"), (200, "NEW PERSON"), (50, "DRUG"), (44, "HOSP LOC")]
summary_rows = []
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    reader = FileReader(conn, dd)
    for fnum, flabel in key_files:
        fd2 = dd.get_file(fnum)
        if not fd2:
            continue
        hits2: dict[float, int] = {}
        n2 = 0
        for entry in reader.iter_entries(fnum, limit=200):
            n2 += 1
            for fn, val in entry.fields.items():
                if val.strip():
                    hits2[fn] = hits2.get(fn, 0) + 1
        for fn, fld in fd2.fields.items():
            pct = 100 * hits2.get(fn, 0) / n2 if n2 else 0
            summary_rows.append(
                {"file": flabel, "field": fld.label, "field_num": fn, "pct": pct}
            )

df_cov = pd.DataFrame(summary_rows)
pivot = df_cov[df_cov["field_num"] == 0.01].pivot(
    index="file", columns="field", values="pct"
)
print(".01 NAME field coverage by file:")
print(pivot.to_string())
df_cov.to_csv(OUT / "phase6_coverage_multi.csv", index=False)
print(f"\nSaved: {OUT / 'phase6_coverage_multi.csv'}")

---
## Phase 7 — Normalization Candidate Identification
**Goal:** Apply rules to schema data from Phases 3–6 to produce a ranked candidate list.

In [ ]:
candidates = []

# Rule 1: Same label, different types
for label, rows in label_groups.items():
    types = set(r["datatype_code"] for r in rows)
    if len(rows) >= 5 and len(types) > 1:
        dominant = collections.Counter(r["datatype_code"] for r in rows).most_common(1)[
            0
        ][0]
        candidates.append(
            {
                "rule": "label_type_conflict",
                "label": label,
                "occurrences": len(rows),
                "types": dict(collections.Counter(r["datatype_code"] for r in rows)),
                "recommended_type": dominant,
                "priority": len(rows),
            }
        )

# Rule 2: Hub file referenced by ≥10 files
for tgt, srcs in inbound.items():
    if len(srcs) >= 10:
        fd_label = next(
            (r["file_label"] for r in schema if r["file_number"] == tgt), "?"
        )
        candidates.append(
            {
                "rule": "hub_file_reference",
                "file": tgt,
                "label": fd_label,
                "source_files": len(srcs),
                "priority": len(srcs),
            }
        )

# Rule 3: DATE field stored as FREE TEXT
for r in schema:
    if "DATE" in r["field_label"].upper() and r["datatype_code"] == "F":
        candidates.append(
            {
                "rule": "date_as_free_text",
                "file": r["file_number"],
                "file_label": r["file_label"],
                "field": r["field_number"],
                "field_label": r["field_label"],
                "package": r["package"],
                "priority": 5,
            }
        )

# Rule 4: Pointer to empty file
volume_map = {num: count for count, num, _ in volume}
for r in schema:
    if r["datatype_code"] == "P" and r["pointer_file"]:
        if volume_map.get(r["pointer_file"], -1) == 0:
            candidates.append(
                {
                    "rule": "pointer_to_empty_file",
                    "file": r["file_number"],
                    "file_label": r["file_label"],
                    "field": r["field_number"],
                    "field_label": r["field_label"],
                    "target_file": r["pointer_file"],
                    "priority": 3,
                }
            )

candidates.sort(key=lambda x: -x["priority"])
out_cands = OUT / "normalization_candidates.json"
out_cands.write_text(json.dumps(candidates, indent=2, default=str))
print(f"Normalization candidates: {len(candidates):,} written to {out_cands}")

### Phase 7 — Visualization

In [ ]:
df_cands = pd.DataFrame(candidates)
rule_counts = df_cands["rule"].value_counts()
rule_colors_7 = {
    "label_type_conflict": "#d62728",
    "hub_file_reference": "#ff7f0e",
    "date_as_free_text": "#9467bd",
    "pointer_to_empty_file": "#1f77b4",
}

fig7, axes7 = plt.subplots(1, 2, figsize=(16, 7))

ax7a = axes7[0]
colors7 = [rule_colors_7.get(r, "gray") for r in rule_counts.index]
bars7 = ax7a.bar(rule_counts.index, rule_counts.values, color=colors7)
ax7a.bar_label(bars7, padding=3)
ax7a.set_xlabel("Rule")
ax7a.set_ylabel("Candidate count")
ax7a.set_title("Normalization Candidates by Rule Type")
ax7a.tick_params(axis="x", rotation=20, labelsize=8)

ax7b = axes7[1]
for rule, grp in df_cands.groupby("rule"):
    x = (
        grp["occurrences"].fillna(grp["priority"])
        if "occurrences" in grp.columns
        else grp["priority"]
    )
    ax7b.scatter(
        x,
        grp["priority"],
        label=rule,
        alpha=0.6,
        color=rule_colors_7.get(rule, "gray"),
        s=40,
    )
ax7b.set_xlabel("Occurrences")
ax7b.set_ylabel("Priority score")
ax7b.set_title("Normalization Candidates — Priority vs Occurrences")
ax7b.legend(fontsize=8)

plt.tight_layout()
fig7.savefig(OUT / "phase7_candidates.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Phase 8 — Normalization Report
**Goal:** Produce the final summary combining all phase outputs.

In [ ]:
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    fi = FileInventory(conn)
    fi.load()

    report = {
        "scope": {
            "total_files": len(dd.list_files()),
            "total_fields": len(schema),
            "total_packages": len(fi.list_packages()),
            "files_with_data": len(volume),
            "files_empty": len(dd.list_files()) - len(volume),
        },
        "volume": {
            "massive_100k_plus": len([v for v in volume if v[0] >= 100_000]),
            "large_10k_100k": len([v for v in volume if 10_000 <= v[0] < 100_000]),
            "medium_1k_10k": len([v for v in volume if 1_000 <= v[0] < 10_000]),
            "small_under_1k": len([v for v in volume if 0 < v[0] < 1_000]),
        },
        "type_distribution": dict(type_counts),
        "pointer_topology": {
            "total_pointer_fields": len(pointer_fields),
            "hub_files_10plus_refs": len(
                [t for t, s in inbound.items() if len(s) >= 10]
            ),
            "top_hubs": [
                {"file": tgt, "inbound_count": len(srcs)}
                for tgt, srcs in sorted(inbound.items(), key=lambda x: -len(x[1]))[:20]
            ],
        },
        "variety": {
            "set_fields_total": len(set_fields),
            "unique_value_sets": len(seen_sets),
            "shared_value_sets_5plus": len(shared),
            "label_type_conflicts": len(
                [c for c in candidates if c["rule"] == "label_type_conflict"]
            ),
            "date_as_free_text": len(
                [c for c in candidates if c["rule"] == "date_as_free_text"]
            ),
        },
        "normalization_candidates_total": len(candidates),
    }

out_report = OUT / "normalization_report.json"
out_report.write_text(json.dumps(report, indent=2, default=str))
print(f"Normalization report written to {out_report}")

### Phase 8 — Summary Dashboard

In [ ]:
from rich import box
from rich.columns import Columns
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()


def stat_panel(title, rows):
    t = Table.grid(padding=(0, 2))
    t.add_column(style="dim")
    t.add_column(style="bold cyan", justify="right")
    for label, value in rows:
        t.add_row(label, str(value))
    return Panel(t, title=f"[bold]{title}[/bold]", border_style="blue")


panels = [
    stat_panel(
        "Scope",
        [
            ("Files", report["scope"]["total_files"]),
            ("Fields", report["scope"]["total_fields"]),
            ("Packages", report["scope"]["total_packages"]),
            ("With data", report["scope"]["files_with_data"]),
            ("Empty", report["scope"]["files_empty"]),
        ],
    ),
    stat_panel(
        "Volume Tiers",
        [
            ("Massive >100K", report["volume"]["massive_100k_plus"]),
            ("Large 10K–100K", report["volume"]["large_10k_100k"]),
            ("Medium 1K–10K", report["volume"]["medium_1k_10k"]),
            ("Small <1K", report["volume"]["small_under_1k"]),
        ],
    ),
    stat_panel(
        "Topology",
        [
            ("Pointer fields", report["pointer_topology"]["total_pointer_fields"]),
            ("Hub files ≥10", report["pointer_topology"]["hub_files_10plus_refs"]),
        ],
    ),
    stat_panel(
        "Variety",
        [
            ("SET fields", report["variety"]["set_fields_total"]),
            ("Unique value sets", report["variety"]["unique_value_sets"]),
            ("Shared sets ≥5", report["variety"]["shared_value_sets_5plus"]),
            ("Label conflicts", report["variety"]["label_type_conflicts"]),
            ("Date-as-text", report["variety"]["date_as_free_text"]),
        ],
    ),
    stat_panel(
        "Normalization",
        [
            ("Total candidates", report["normalization_candidates_total"]),
        ],
    ),
]
console.print()
console.print(Columns(panels, equal=True))

# Top hubs table
hub_t = Table(title="Top Hub Files", box=box.SIMPLE)
hub_t.add_column("File #", style="cyan", justify="right")
hub_t.add_column("Label", style="white")
hub_t.add_column("Inbound", style="yellow", justify="right")
with YdbConnection.connect() as conn:
    dd = DataDictionary(conn)
    for h in report["pointer_topology"]["top_hubs"][:10]:
        fd_h = dd.get_file(h["file"])
        hub_t.add_row(
            str(h["file"]), fd_h.label if fd_h else "?", str(h["inbound_count"])
        )
console.print(hub_t)
print(f"\nAll outputs in: {OUT}")